# PyTorch Tensor Indexing — 110 Progressive Exercises


> **Virgin retry copy:** Answer cells are intentionally blank and all saved execution outputs are cleared. Copy this file before a fresh attempt; do not record ongoing work in the virgin file.

This workbook builds from one-dimensional scalar indexing to batched loss extraction, storage semantics, gradients, and vectorized capstones. The original scaffold contains no official answer key. This virgin copy contains no learner answers; continue writing code only in student-answer cells.

## How to use the workbook

1. Run the supplied setup cell once.
2. Read one exercise's Markdown prompt.
3. Type code only in its student-answer cell and keep the required variable name.
4. Run the supplied test directly below it.
5. Move on only after it prints `PASS`.

Tests check value, shape, dtype, and—in selected exercises—storage aliasing. A failing undefined-name message means the answer cell has not created the required variable. Assignment exercises work on clones; if a test says `a` changed, rerun setup and repair the exercise so the original input remains untouched.

The sequence is deliberately progressive. Avoid loops whenever a prompt says vectorized; the point is to express the selection with tensor indices.


## Notebook map

1. Axes, scalar indices, rows, and columns — Exercises 001–010
2. Slices and shape preservation — Exercises 011–020
3. Higher-rank tensors, ellipsis, and new axes — Exercises 021–030
4. Advanced integer indexing and broadcasting — Exercises 031–045
5. Boolean masks and coordinate discovery — Exercises 046–058
6. Indexed assignment, broadcasting, views, and copies — Exercises 059–070
7. Dedicated PyTorch indexing operators — Exercises 071–090
8. Machine-learning indexing patterns — Exercises 091–100
9. Storage and autograd semantics — Exercises 101–106
10. Vectorized mastery capstones — Exercises 107–110


In [ ]:
# Supplied setup: run this cell before starting or after restarting the kernel.
import torch
import torch.nn.functional as F

# The row axis is axis 0; the column axis is axis 1.
a = torch.tensor([
    [0, 1, 2, 3, 4, 5],
    [10, 11, 12, 13, 14, 15],
    [20, 21, 22, 23, 24, 25],
    [30, 31, 32, 33, 34, 35],
])
v = torch.tensor([5, 10, 15, 20, 25, 30, 35, 40])

# Higher-rank inputs make each axis visually traceable.
cube = torch.arange(24).reshape(2, 3, 4)
images = torch.arange(120).reshape(2, 3, 4, 5)

# Reusable integer and Boolean indices.
row_ids = torch.tensor([3, 0, 2])
col_ids = torch.tensor([1, 5, 3])
target_cols = torch.tensor([2, 0, 5, 1])
row_mask = torch.tensor([True, False, True, False])
col_mask = torch.tensor([False, True, False, False, True, True])

# Four training examples (rows) and five class candidates (columns).
scores = torch.tensor([
    [0.2, -1.0, 1.3, 2.2, 0.0],
    [2.0, 0.5, -0.3, 1.0, 1.1],
    [-1.0, 0.0, 0.1, -0.1, 3.0],
    [0.3, 2.4, 1.4, -2.0, 0.7],
])
class_targets = torch.tensor([3, 4, 4, 2])
class_weights = torch.tensor([1.0, 1.5, 0.5, 2.0, 3.0])

# Two sequences, three positions per sequence, and five vocabulary candidates.
token_logits = torch.linspace(-2.0, 2.0, steps=30).reshape(2, 3, 5)
token_targets = torch.tensor([[0, 2, 4], [1, 3, 0]])
embedding_table = torch.arange(24, dtype=torch.float32).reshape(8, 3) / 10
token_ids = torch.tensor([[2, 5, 1, 0], [7, 3, 3, 4]])

# Inputs for later indexing operators and capstones.
gather_cols = torch.tensor([[0, 5], [1, 4], [2, 3], [5, 0]])
cube_feature_idx = torch.tensor([
    [[0, 3], [1, 2], [2, 0]],
    [[3, 1], [0, 2], [1, 3]],
])
score_order = scores.argsort(dim=1, descending=True)
flat_take = torch.tensor([0, 7, 14, 23])
batch_mats = torch.arange(32).reshape(2, 4, 4)
padded = torch.tensor([
    [4, 5, 6, 0, 0],
    [7, 8, 9, 10, 0],
    [11, 12, 0, 0, 0],
])
lengths = torch.tensor([3, 4, 2])
true_classes = torch.tensor([0, 1, 2, 1, 0, 2, 2])
pred_classes = torch.tensor([0, 2, 2, 1, 1, 0, 2])

# Keep an untouched reference so assignment exercises detect accidental input mutation.
_A_CANONICAL = a.clone()
_MISSING = object()


def _check_tensor(name, expected):
    """Check presence, tensor type, shape, dtype, and values."""
    actual = globals().get(name, _MISSING)
    assert actual is not _MISSING, f"Define `{name}` in the exercise cell first."
    assert isinstance(actual, torch.Tensor), f"`{name}` must be a torch.Tensor."
    assert actual.shape == expected.shape, (
        f"`{name}` has shape {tuple(actual.shape)}; expected {tuple(expected.shape)}."
    )
    assert actual.dtype == expected.dtype, (
        f"`{name}` has dtype {actual.dtype}; expected {expected.dtype}."
    )
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    print(f"PASS: {name}")


def _check_value(name, expected):
    """Check a required non-tensor Python value."""
    actual = globals().get(name, _MISSING)
    assert actual is not _MISSING, f"Define `{name}` in the exercise cell first."
    assert type(actual) is type(expected), (
        f"`{name}` has type {type(actual).__name__}; expected {type(expected).__name__}."
    )
    assert actual == expected, f"`{name}` has value {actual!r}; expected {expected!r}."
    print(f"PASS: {name}")


def _check_storage(name, other, should_share):
    """Check whether two nonempty CPU tensors share the same storage."""
    actual = globals().get(name, _MISSING)
    assert actual is not _MISSING, f"Define `{name}` in the exercise cell first."
    assert isinstance(actual, torch.Tensor), f"`{name}` must be a torch.Tensor."
    shares = actual.untyped_storage().data_ptr() == other.untyped_storage().data_ptr()
    assert shares is should_share, (
        f"Storage-sharing check for `{name}` was {shares}; expected {should_share}."
    )
    print(f"PASS: {name} storage semantics")


print("Setup complete. Start at Exercise 001 and run each supplied test after your answer.")


## 1. Axes, scalar indices, rows, and columns

A rank-2 tensor uses axis 0 for rows and axis 1 for columns. Integer indexing chooses one position on an axis and removes that axis from the result. Begin by predicting both the values and the result rank before running each test.


### Exercise 001 — Top-left scalar

**Purpose:** Read one scalar from a rank-2 tensor.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select the element at row 0, column 0.

**Ingredients:** Two integer indices inside one pair of brackets.

**Output:** Assign the result to `ex001`. Produce a rank-0 integer tensor.

**Next concept:** Interior scalar.


In [ ]:
# Exercise 001: assign your result to `ex001`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex001", torch.tensor([0], dtype=torch.int64).reshape(()))


### Exercise 002 — Interior scalar

**Purpose:** Move independently along the row and column axes.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select row 2, column 4.

**Ingredients:** Two nonnegative integer indices.

**Output:** Assign the result to `ex002`. Produce a rank-0 integer tensor.

**Next concept:** Negative indices.


In [ ]:
# Exercise 002: assign your result to `ex002`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex002", torch.tensor([24], dtype=torch.int64).reshape(()))


### Exercise 003 — Negative indices

**Purpose:** Use offsets measured from the end of each axis.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select the final row and final column without writing their positive positions.

**Ingredients:** Negative integer indexing on both axes.

**Output:** Assign the result to `ex003`. Produce a rank-0 integer tensor.

**Next concept:** Implicit full row.


In [ ]:
# Exercise 003: assign your result to `ex003`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex003", torch.tensor([35], dtype=torch.int64).reshape(()))


### Exercise 004 — Implicit full row

**Purpose:** See that one integer index applies to axis 0.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select the first row using only one index.

**Ingredients:** A single integer index.

**Output:** Assign the result to `ex004`. Produce a rank-1 tensor of length 6.

**Next concept:** Explicit full row.


In [ ]:
# Exercise 004: assign your result to `ex004`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex004", torch.tensor([0, 1, 2, 3, 4, 5], dtype=torch.int64).reshape((6,)))


### Exercise 005 — Explicit full row

**Purpose:** Use a full slice to state that every column is retained.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select row 2 and all of its columns explicitly.

**Ingredients:** An integer on axis 0 and `:` on axis 1.

**Output:** Assign the result to `ex005`. Produce a rank-1 tensor of length 6.

**Next concept:** Full column.


In [ ]:
# Exercise 005: assign your result to `ex005`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex005", torch.tensor([20, 21, 22, 23, 24, 25], dtype=torch.int64).reshape((6,)))


### Exercise 006 — Full column

**Purpose:** Select down axis 0 while fixing axis 1.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select every row from column 1.

**Ingredients:** A full slice on axis 0 and an integer on axis 1.

**Output:** Assign the result to `ex006`. Produce a rank-1 tensor of length 4.

**Next concept:** Final column.


In [ ]:
# Exercise 006: assign your result to `ex006`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex006", torch.tensor([1, 11, 21, 31], dtype=torch.int64).reshape((4,)))


### Exercise 007 — Final column

**Purpose:** Combine a full slice with a negative column index.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select the final column from every row.

**Ingredients:** A full row slice and a negative integer column index.

**Output:** Assign the result to `ex007`. Produce a rank-1 tensor of length 4.

**Next concept:** Index a vector.


In [ ]:
# Exercise 007: assign your result to `ex007`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex007", torch.tensor([5, 15, 25, 35], dtype=torch.int64).reshape((4,)))


### Exercise 008 — Index a vector

**Purpose:** Apply integer indexing to a rank-1 tensor.

**Inputs:** `v`, shape `(8,)`.

**Task:** Select the item at position 3.

**Ingredients:** One nonnegative integer index.

**Output:** Assign the result to `ex008`. Produce a rank-0 integer tensor.

**Next concept:** Negative vector index.


In [ ]:
# Exercise 008: assign your result to `ex008`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex008", torch.tensor([20], dtype=torch.int64).reshape(()))


### Exercise 009 — Negative vector index

**Purpose:** Count backward in a rank-1 tensor.

**Inputs:** `v`, shape `(8,)`.

**Task:** Select the third item from the end.

**Ingredients:** One negative integer index.

**Output:** Assign the result to `ex009`. Produce a rank-0 integer tensor.

**Next concept:** Tensor scalar to Python scalar.


In [ ]:
# Exercise 009: assign your result to `ex009`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex009", torch.tensor([30], dtype=torch.int64).reshape(()))


### Exercise 010 — Tensor scalar to Python scalar

**Purpose:** Distinguish a rank-0 tensor from a native Python number.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select row 1, column 2, then convert the selected tensor scalar to a Python integer.

**Ingredients:** Integer indexing followed by the scalar conversion method `Tensor.item()`.

**Output:** Assign the result to `ex010`. Produce a Python `int`, not a tensor.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 010: assign your result to `ex010`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_value("ex010", 12)


## 2. Slices and shape preservation

A slice has `start:stop:step` semantics. The stop is excluded. Unlike an integer index, a slice preserves its axis—even when it selects only one position. PyTorch does not support a negative slice step, so reversal appears later through integer indices or `torch.flip`.


### Exercise 011 — Prefix of columns

**Purpose:** Select a rectangular prefix.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep every row and the first three columns.

**Ingredients:** A full row slice and a bounded column slice; remember that stop is excluded.

**Output:** Assign the result to `ex011`. Produce shape `(4, 3)`.

**Next concept:** Column suffix.


In [ ]:
# Exercise 011: assign your result to `ex011`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex011", torch.tensor([0, 1, 2, 10, 11, 12, 20, 21, 22, 30, 31, 32], dtype=torch.int64).reshape((4, 3)))


### Exercise 012 — Column suffix

**Purpose:** Use an omitted slice stop.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep every row and columns from position 2 through the end.

**Ingredients:** A column slice with a start and omitted stop.

**Output:** Assign the result to `ex012`. Produce shape `(4, 4)`.

**Next concept:** Interior rectangle.


In [ ]:
# Exercise 012: assign your result to `ex012`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex012", torch.tensor([2, 3, 4, 5, 12, 13, 14, 15, 22, 23, 24, 25, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 4)))


### Exercise 013 — Interior rectangle

**Purpose:** Slice both axes at once.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep rows 1 and 2 and columns 2, 3, and 4.

**Ingredients:** Two bounded slices with exclusive stops.

**Output:** Assign the result to `ex013`. Produce shape `(2, 3)`.

**Next concept:** Even-positioned columns.


In [ ]:
# Exercise 013: assign your result to `ex013`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex013", torch.tensor([12, 13, 14, 22, 23, 24], dtype=torch.int64).reshape((2, 3)))


### Exercise 014 — Even-positioned columns

**Purpose:** Use a slice step.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep every row and columns 0, 2, and 4.

**Ingredients:** A column slice with step 2.

**Output:** Assign the result to `ex014`. Produce shape `(4, 3)`.

**Next concept:** Odd-positioned columns.


In [ ]:
# Exercise 014: assign your result to `ex014`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex014", torch.tensor([0, 2, 4, 10, 12, 14, 20, 22, 24, 30, 32, 34], dtype=torch.int64).reshape((4, 3)))


### Exercise 015 — Odd-positioned columns

**Purpose:** Offset a strided slice.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep every row and columns 1, 3, and 5.

**Ingredients:** A column slice starting at 1 with step 2.

**Output:** Assign the result to `ex015`. Produce shape `(4, 3)`.

**Next concept:** Stride both axes.


In [ ]:
# Exercise 015: assign your result to `ex015`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex015", torch.tensor([1, 3, 5, 11, 13, 15, 21, 23, 25, 31, 33, 35], dtype=torch.int64).reshape((4, 3)))


### Exercise 016 — Stride both axes

**Purpose:** Combine independent steps on two axes.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep rows 0 and 2 and columns 0 and 3.

**Ingredients:** A row slice with step 2 and a column slice with step 3.

**Output:** Assign the result to `ex016`. Produce shape `(2, 2)`.

**Next concept:** Last two rows.


In [ ]:
# Exercise 016: assign your result to `ex016`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex016", torch.tensor([0, 3, 20, 23], dtype=torch.int64).reshape((2, 2)))


### Exercise 017 — Last two rows

**Purpose:** Use a negative start with an omitted stop.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep the final two rows and every column.

**Ingredients:** A row suffix slice and a full column slice.

**Output:** Assign the result to `ex017`. Produce shape `(2, 6)`.

**Next concept:** Exclude the final column.


In [ ]:
# Exercise 017: assign your result to `ex017`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex017", torch.tensor([20, 21, 22, 23, 24, 25, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((2, 6)))


### Exercise 018 — Exclude the final column

**Purpose:** Use a negative exclusive stop.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep every row and every column except the final one.

**Ingredients:** A column slice whose stop is `-1`.

**Output:** Assign the result to `ex018`. Produce shape `(4, 5)`.

**Next concept:** Empty slice.


In [ ]:
# Exercise 018: assign your result to `ex018`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex018", torch.tensor([0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 20, 21, 22, 23, 24, 30, 31, 32, 33, 34], dtype=torch.int64).reshape((4, 5)))


### Exercise 019 — Empty slice

**Purpose:** Recognize that valid slices may select zero positions.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select every row and the column interval from 2 up to, but not including, 2.

**Ingredients:** Equal slice start and stop.

**Output:** Assign the result to `ex019`. Produce an empty tensor with shape `(4, 0)`.

**Next concept:** One-row slice.


In [ ]:
# Exercise 019: assign your result to `ex019`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex019", torch.tensor([], dtype=torch.int64).reshape((4, 0)))


### Exercise 020 — One-row slice

**Purpose:** Preserve the row axis with a length-one slice.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select only row 1 while keeping the result rank 2.

**Ingredients:** A row slice of length one, not an integer row index.

**Output:** Assign the result to `ex020`. Produce shape `(1, 6)`.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 020: assign your result to `ex020`.
# Write your solution below this comment, then run the supplied test cell.
# ex020 = a[1, :] # incorrect!


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex020", torch.tensor([10, 11, 12, 13, 14, 15], dtype=torch.int64).reshape((1, 6)))


## 3. Higher-rank tensors, ellipsis, and new axes

For `cube`, the axes are batch, channel, and feature. For `images`, they are batch, channel, height, and width. An ellipsis stands for as many untouched axes as needed. `None` inserts a singleton axis without consuming an input axis.


### Exercise 021 — Rank-3 scalar

**Purpose:** Address all three axes of a tensor.

**Inputs:** `cube`, shape `(2, 3, 4)`.

**Task:** Select batch 1, channel 2, feature 3.

**Ingredients:** Three integer indices.

**Output:** Assign the result to `ex021`. Produce a rank-0 integer tensor.

**Next concept:** One batch plane.


In [ ]:
# Exercise 021: assign your result to `ex021`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex021", torch.tensor([23], dtype=torch.int64).reshape(()))


### Exercise 022 — One batch plane

**Purpose:** Remove only the batch axis.

**Inputs:** `cube`, shape `(2, 3, 4)`.

**Task:** Select batch 0 and retain all channels and features.

**Ingredients:** One integer index; omitted trailing axes are retained.

**Output:** Assign the result to `ex022`. Produce shape `(3, 4)`.

**Next concept:** One channel across batches.


In [ ]:
# Exercise 022: assign your result to `ex022`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex022", torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], dtype=torch.int64).reshape((3, 4)))


### Exercise 023 — One channel across batches

**Purpose:** Fix the middle axis while retaining the outer axes.

**Inputs:** `cube`, shape `(2, 3, 4)`.

**Task:** Select channel 1 for every batch and every feature.

**Ingredients:** Full slices around one integer middle-axis index.

**Output:** Assign the result to `ex023`. Produce shape `(2, 4)`.

**Next concept:** One feature across batches and channels.


In [ ]:
# Exercise 023: assign your result to `ex023`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex023", torch.tensor([4, 5, 6, 7, 16, 17, 18, 19], dtype=torch.int64).reshape((2, 4)))


### Exercise 024 — One feature across batches and channels

**Purpose:** Fix the final axis.

**Inputs:** `cube`, shape `(2, 3, 4)`.

**Task:** Select feature 2 for every batch and channel.

**Ingredients:** Full slices on the first two axes and an integer on the last.

**Output:** Assign the result to `ex024`. Produce shape `(2, 3)`.

**Next concept:** Ellipsis before a feature.


In [ ]:
# Exercise 024: assign your result to `ex024`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex024", torch.tensor([2, 6, 10, 14, 18, 22], dtype=torch.int64).reshape((2, 3)))


### Exercise 025 — Ellipsis before a feature

**Purpose:** Replace repeated full slices with an ellipsis.

**Inputs:** `cube`, shape `(2, 3, 4)`.

**Task:** Repeat Exercise 024, but use an ellipsis in the index.

**Ingredients:** The `...` index followed by one integer.

**Output:** Assign the result to `ex025`. Produce shape `(2, 3)`.

**Next concept:** Ellipsis after a batch.


In [ ]:
# Exercise 025: assign your result to `ex025`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex025", torch.tensor([2, 6, 10, 14, 18, 22], dtype=torch.int64).reshape((2, 3)))


### Exercise 026 — Ellipsis after a batch

**Purpose:** Retain all trailing axes concisely.

**Inputs:** `cube`, shape `(2, 3, 4)`.

**Task:** Select the final batch and retain every remaining axis using an ellipsis.

**Ingredients:** A negative integer index followed by `...`.

**Output:** Assign the result to `ex026`. Produce shape `(3, 4)`.

**Next concept:** Leading singleton axis.


In [ ]:
# Exercise 026: assign your result to `ex026`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex026", torch.tensor([12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23], dtype=torch.int64).reshape((3, 4)))


### Exercise 027 — Leading singleton axis

**Purpose:** Insert an axis before existing data.

**Inputs:** `v`, shape `(8,)`.

**Task:** Add a size-1 axis at the front using indexing syntax only.

**Ingredients:** `None` (equivalently `torch.newaxis`) before the full vector slice.

**Output:** Assign the result to `ex027`. Produce shape `(1, 8)`.

**Next concept:** Trailing singleton axis.


In [ ]:
# Exercise 027: assign your result to `ex027`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex027", torch.tensor([5, 10, 15, 20, 25, 30, 35, 40], dtype=torch.int64).reshape((1, 8)))


### Exercise 028 — Trailing singleton axis

**Purpose:** Turn a vector into a column-shaped tensor.

**Inputs:** `v`, shape `(8,)`.

**Task:** Add a size-1 axis after the existing vector axis using indexing syntax only.

**Ingredients:** A full vector slice followed by `None`.

**Output:** Assign the result to `ex028`. Produce shape `(8, 1)`.

**Next concept:** Middle singleton axis.


In [ ]:
# Exercise 028: assign your result to `ex028`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex028", torch.tensor([5, 10, 15, 20, 25, 30, 35, 40], dtype=torch.int64).reshape((8, 1)))


### Exercise 029 — Middle singleton axis

**Purpose:** Insert an axis between two retained axes.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Add a singleton axis between rows and columns.

**Ingredients:** A full row slice, `None`, and a full column slice.

**Output:** Assign the result to `ex029`. Produce shape `(4, 1, 6)`.

**Next concept:** Image channel crop with preserved channel axis.


In [ ]:
# Exercise 029: assign your result to `ex029`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex029", torch.tensor([0, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15, 20, 21, 22, 23, 24, 25, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 1, 6)))


### Exercise 030 — Image channel crop with preserved channel axis

**Purpose:** Combine four-axis slicing while preserving every axis.

**Inputs:** `images`, shape `(2, 3, 4, 5)` ordered as batch, channel, height, width.

**Task:** For both batches, keep only channel 1 as a length-one axis, heights 1 and 2, and widths 2, 3, and 4.

**Ingredients:** Four slices; use a length-one channel slice rather than an integer.

**Output:** Assign the result to `ex030`. Produce shape `(2, 1, 2, 3)`.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 030: assign your result to `ex030`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex030", torch.tensor([27, 28, 29, 32, 33, 34, 87, 88, 89, 92, 93, 94], dtype=torch.int64).reshape((2, 1, 2, 3)))


## 4. Advanced integer indexing and broadcasting

A list or non-scalar integer tensor is an advanced index. Multiple advanced indices are paired element by element after broadcasting; they do not automatically form a Cartesian product. This is why `a[range(n), target_cols]` selects one column per row, while `a[:, target_cols]` applies the entire column list to every row.


### Exercise 031 — Reordered rows with duplicates

**Purpose:** Use an advanced row index to reorder and repeat data.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select rows in the order 3, 0, 3 using a Python list.

**Ingredients:** A Python list used on axis 0.

**Output:** Assign the result to `ex031`. Produce shape `(3, 6)`.

**Next concept:** Reordered columns with an index tensor.


In [ ]:
# Exercise 031: assign your result to `ex031`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex031", torch.tensor([30, 31, 32, 33, 34, 35, 0, 1, 2, 3, 4, 5, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((3, 6)))


### Exercise 032 — Reordered columns with an index tensor

**Purpose:** Apply one integer tensor to the column axis.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** For every row, select columns in the order 5, 2, 0.

**Ingredients:** A full row slice and a rank-1 `torch.long` column index.

**Output:** Assign the result to `ex032`. Produce shape `(4, 3)`.

**Next concept:** Paired coordinates.


In [ ]:
# Exercise 032: assign your result to `ex032`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex032", torch.tensor([5, 2, 0, 15, 12, 10, 25, 22, 20, 35, 32, 30], dtype=torch.int64).reshape((4, 3)))


### Exercise 033 — Paired coordinates

**Purpose:** Pair one row index with one column index.

**Inputs:** `a`, `row_ids = [3, 0, 2]`, and `col_ids = [1, 5, 3]`.

**Task:** Select the three coordinates formed by pairing corresponding entries of `row_ids` and `col_ids`.

**Ingredients:** Two rank-1 integer tensors in the same index.

#### How paired indexing behaves

In `a[row_ids, col_ids]`, both index tensors have shape `(3,)`. PyTorch pairs entries at matching positions rather than selecting every row-column combination:

- index position 0 selects coordinate `(3, 1)`, whose value is `31`;
- index position 1 selects coordinate `(0, 5)`, whose value is `5`;
- index position 2 selects coordinate `(2, 3)`, whose value is `23`.

Conceptually, output entry `i` is `a[row_ids[i], col_ids[i]]`. There are three coordinate pairs, so the result is `tensor([31, 5, 23])` with shape `(3,)`. This is different from `a[:, col_ids]`, which applies all three column indices to every row and produces shape `(4, 3)`.

**Output:** Assign the result to `ex033`. Produce shape `(3,)`, one value per coordinate pair.

**Next concept:** One target column per row.


In [ ]:
# Exercise 033: assign your result to `ex033`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex033", torch.tensor([31, 5, 23], dtype=torch.int64).reshape((3,)))


### Exercise 034 — One target column per row

**Purpose:** Master the `range(n)` pattern behind classification loss indexing.

**Inputs:** `a`, four rows, and `target_cols`, one column index per row.

**Task:** Select one value from each row: row 0 with `target_cols[0]`, row 1 with `target_cols[1]`, and so on. Use `range` for the row axis.

**Ingredients:** A Python `range` and a rank-1 integer tensor as paired advanced indices.

**Output:** Assign the result to `ex034`. Produce shape `(4,)`.

**Next concept:** Every target column for every row.


In [ ]:
# Exercise 034: assign your result to `ex034`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex034", torch.tensor([2, 10, 25, 31], dtype=torch.int64).reshape((4,)))


### Exercise 035 — Every target column for every row

**Purpose:** Contrast Cartesian-like column selection with paired indexing.

**Inputs:** `a` and `target_cols`.

**Task:** Apply the complete `target_cols` sequence to every row by keeping the row axis as a full slice.

**Ingredients:** A full row slice plus one rank-1 advanced column index.

**Output:** Assign the result to `ex035`. Produce shape `(4, 4)`, not `(4,)`.

**Next concept:** Cartesian submatrix by broadcasting.


In [ ]:
# Exercise 035: assign your result to `ex035`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex035", torch.tensor([2, 0, 5, 1, 12, 10, 15, 11, 22, 20, 25, 21, 32, 30, 35, 31], dtype=torch.int64).reshape((4, 4)))


### Exercise 036 — Cartesian submatrix by broadcasting

**Purpose:** Create every combination of selected rows and columns.

**Inputs:** `a`; desired rows are 0 and 2; desired columns are 1, 4, and 5.

**Task:** Use broadcastable row and column index tensors to extract the 2-by-3 Cartesian submatrix in one indexing operation.

**Ingredients:** Insert singleton axes into the row and column index tensors so their shapes broadcast.

**Output:** Assign the result to `ex036`. Produce shape `(2, 3)`.

**Next concept:** Cartesian submatrix with meshgrid.


In [ ]:
# Exercise 036: assign your result to `ex036`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex036", torch.tensor([1, 4, 5, 21, 24, 25], dtype=torch.int64).reshape((2, 3)))


### Exercise 037 — Cartesian submatrix with meshgrid

**Purpose:** Generate Cartesian coordinate grids explicitly.

**Inputs:** `a`; desired rows are 0 and 2; desired columns are 1, 4, and 5.

**Task:** Create row and column grids with `torch.meshgrid` using matrix (`ij`) indexing, then use them together.

**Ingredients:** `torch.meshgrid(..., indexing="ij")` and paired advanced indexing.

**Output:** Assign the result to `ex037`. Produce shape `(2, 3)`.

**Next concept:** Reverse rows without a negative slice step.


In [ ]:
# Exercise 037: assign your result to `ex037`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex037", torch.tensor([1, 4, 5, 21, 24, 25], dtype=torch.int64).reshape((2, 3)))


### Exercise 038 — Reverse rows without a negative slice step

**Purpose:** Reverse one axis using explicit integer positions.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Return all rows in reverse order while leaving columns unchanged. Do not use `torch.flip` in this exercise.

**Ingredients:** A descending rank-1 integer index; negative slice steps are unsupported.

**Output:** Assign the result to `ex038`. Produce shape `(4, 6)`.

**Next concept:** Repeat columns.


In [ ]:
# Exercise 038: assign your result to `ex038`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex038", torch.tensor([30, 31, 32, 33, 34, 35, 20, 21, 22, 23, 24, 25, 10, 11, 12, 13, 14, 15, 0, 1, 2, 3, 4, 5], dtype=torch.int64).reshape((4, 6)))


### Exercise 039 — Repeat columns

**Purpose:** See that advanced indices may contain duplicates.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select columns 1, 1, and 4 for every row, preserving that order.

**Ingredients:** A rank-1 integer column index with a repeated position.

**Output:** Assign the result to `ex039`. Produce shape `(4, 3)`.

**Next concept:** Main diagonal through paired indices.


In [ ]:
# Exercise 039: assign your result to `ex039`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex039", torch.tensor([1, 1, 4, 11, 11, 14, 21, 21, 24, 31, 31, 34], dtype=torch.int64).reshape((4, 3)))


### Exercise 040 — Main diagonal through paired indices

**Purpose:** Extract matching row and column positions without a loop.

**Inputs:** `a`; use only its first four columns.

**Task:** Select coordinates (0,0), (1,1), (2,2), and (3,3) with paired indices.

**Ingredients:** One shared rank-1 index for both axes.

**Output:** Assign the result to `ex040`. Produce shape `(4,)`.

**Next concept:** Anti-diagonal through paired indices.


In [ ]:
# Exercise 040: assign your result to `ex040`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex040", torch.tensor([0, 11, 22, 33], dtype=torch.int64).reshape((4,)))


### Exercise 041 — Anti-diagonal through paired indices

**Purpose:** Pair ascending rows with descending columns.

**Inputs:** `a`; use columns 0 through 3.

**Task:** Select coordinates (0,3), (1,2), (2,1), and (3,0) without a loop.

**Ingredients:** An ascending row index and a descending column index.

**Output:** Assign the result to `ex041`. Produce shape `(4,)`.

**Next concept:** Advanced rows plus a basic column slice.


In [ ]:
# Exercise 041: assign your result to `ex041`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex041", torch.tensor([3, 12, 21, 30], dtype=torch.int64).reshape((4,)))


### Exercise 042 — Advanced rows plus a basic column slice

**Purpose:** Mix advanced and basic indexing in one expression.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select rows 0 and 2, then columns 2, 3, and 4.

**Ingredients:** A rank-1 advanced row index and a basic column slice.

**Output:** Assign the result to `ex042`. Produce shape `(2, 3)`.

**Next concept:** A rank-2 row index.


In [ ]:
# Exercise 042: assign your result to `ex042`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex042", torch.tensor([2, 3, 4, 22, 23, 24], dtype=torch.int64).reshape((2, 3)))


### Exercise 043 — A rank-2 row index

**Purpose:** Observe that the advanced index shape appears in the result.

**Inputs:** `a` and row index `[[0, 1], [3, 2]]`.

**Task:** Use that rank-2 integer tensor to select rows from `a`.

**Ingredients:** A rank-2 `torch.long` index on axis 0.

**Output:** Assign the result to `ex043`. Produce shape `(2, 2, 6)`.

**Next concept:** Separated advanced indices in rank 3.


In [ ]:
# Exercise 043: assign your result to `ex043`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex043", torch.tensor([0, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15, 30, 31, 32, 33, 34, 35, 20, 21, 22, 23, 24, 25], dtype=torch.int64).reshape((2, 2, 6)))


### Exercise 044 — Separated advanced indices in rank 3

**Purpose:** Pair advanced indices on nonadjacent axes while retaining a basic-sliced axis.

**Inputs:** `cube`, shape `(2, 3, 4)`.

**Task:** Pair batch indices `[0, 1]` with feature indices `[1, 3]` and retain every channel between them.

**Ingredients:** Advanced indices on axes 0 and 2 with a full basic slice on axis 1.

**Output:** Assign the result to `ex044`. Produce shape `(2, 3)`; each output row corresponds to one batch-feature pair.

**Next concept:** Broadcast paired coordinate arrays.


In [ ]:
# Exercise 044: assign your result to `ex044`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex044", torch.tensor([1, 5, 9, 15, 19, 23], dtype=torch.int64).reshape((2, 3)))


### Exercise 045 — Broadcast paired coordinate arrays

**Purpose:** Broadcast differently shaped advanced indices.

**Inputs:** `a`; rows are `[[0], [2], [3]]` and columns are `[[1, 4]]`.

**Task:** Use the two already-shaped index tensors together to select all six broadcast coordinate pairs.

**Ingredients:** Advanced index broadcasting from shapes `(3, 1)` and `(1, 2)`.

**Output:** Assign the result to `ex045`. Produce shape `(3, 2)`.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 045: assign your result to `ex045`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex045", torch.tensor([1, 4, 21, 24, 31, 34], dtype=torch.int64).reshape((3, 2)))


## 5. Boolean masks and coordinate discovery

A Boolean index answers which positions to retain. A full elementwise mask returns matching values as a rank-1 tensor in row-major order. A one-dimensional mask applied to one axis retains that axis's matching rows or columns. Combine conditions with `&`, `|`, and `~`, and parenthesize each comparison.


### Exercise 046 — Threshold a vector

**Purpose:** Filter a rank-1 tensor by a comparison.

**Inputs:** `v`, shape `(8,)`.

**Task:** Keep only values strictly greater than 20.

**Ingredients:** Create an elementwise Boolean mask with `>` and use it as the index.

**Output:** Assign the result to `ex046`. Produce a rank-1 integer tensor.

**Next concept:** Full element mask.


In [ ]:
# Exercise 046: assign your result to `ex046`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex046", torch.tensor([25, 30, 35, 40], dtype=torch.int64).reshape((4,)))


### Exercise 047 — Full element mask

**Purpose:** Observe that a same-shape mask flattens selected values.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep every even-valued element.

**Ingredients:** Remainder, equality, and a full Boolean mask.

**Output:** Assign the result to `ex047`. Produce a rank-1 tensor in row-major selection order.

**Next concept:** Boolean row mask.


In [ ]:
# Exercise 047: assign your result to `ex047`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex047", torch.tensor([0, 2, 4, 10, 12, 14, 20, 22, 24, 30, 32, 34], dtype=torch.int64).reshape((12,)))


### Exercise 048 — Boolean row mask

**Purpose:** Filter one axis with a one-dimensional mask.

**Inputs:** `a` and `row_mask`, shape `(4,)`.

**Task:** Keep the rows whose entries in `row_mask` are true.

**Ingredients:** Use the Boolean mask on axis 0.

**Output:** Assign the result to `ex048`. Produce shape `(2, 6)`.

**Next concept:** Boolean column mask.


In [ ]:
# Exercise 048: assign your result to `ex048`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex048", torch.tensor([0, 1, 2, 3, 4, 5, 20, 21, 22, 23, 24, 25], dtype=torch.int64).reshape((2, 6)))


### Exercise 049 — Boolean column mask

**Purpose:** Filter columns while retaining every row.

**Inputs:** `a` and `col_mask`, shape `(6,)`.

**Task:** Keep columns selected by `col_mask` for every row.

**Ingredients:** A full row slice and a Boolean mask on axis 1.

**Output:** Assign the result to `ex049`. Produce shape `(4, 3)`.

**Next concept:** Threshold a matrix.


In [ ]:
# Exercise 049: assign your result to `ex049`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex049", torch.tensor([1, 4, 5, 11, 14, 15, 21, 24, 25, 31, 34, 35], dtype=torch.int64).reshape((4, 3)))


### Exercise 050 — Threshold a matrix

**Purpose:** Filter all matrix elements with one comparison.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep values greater than or equal to 23.

**Ingredients:** A same-shape comparison mask.

**Output:** Assign the result to `ex050`. Produce a rank-1 tensor.

**Next concept:** Conjunction of three conditions.


In [ ]:
# Exercise 050: assign your result to `ex050`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex050", torch.tensor([23, 24, 25, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((9,)))


### Exercise 051 — Conjunction of three conditions

**Purpose:** Intersect multiple elementwise criteria.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep values greater than 10, less than 30, and even.

**Ingredients:** Parenthesized comparisons combined with `&`.

**Output:** Assign the result to `ex051`. Produce a rank-1 tensor.

**Next concept:** Disjunction of conditions.


In [ ]:
# Exercise 051: assign your result to `ex051`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex051", torch.tensor([12, 14, 20, 22, 24], dtype=torch.int64).reshape((5,)))


### Exercise 052 — Disjunction of conditions

**Purpose:** Select values satisfying either boundary condition.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep values below 3 or above 32.

**Ingredients:** Parenthesized comparisons combined with `|`.

**Output:** Assign the result to `ex052`. Produce a rank-1 tensor.

**Next concept:** Invert a row mask.


In [ ]:
# Exercise 052: assign your result to `ex052`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex052", torch.tensor([0, 1, 2, 33, 34, 35], dtype=torch.int64).reshape((6,)))


### Exercise 053 — Invert a row mask

**Purpose:** Select the complement of an existing mask.

**Inputs:** `a` and `row_mask`.

**Task:** Keep precisely the rows that `row_mask` excludes.

**Ingredients:** Boolean inversion with `~` on the row mask.

**Output:** Assign the result to `ex053`. Produce shape `(2, 6)`.

**Next concept:** Membership mask.


In [ ]:
# Exercise 053: assign your result to `ex053`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex053", torch.tensor([10, 11, 12, 13, 14, 15, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((2, 6)))


### Exercise 054 — Membership mask

**Purpose:** Filter values by membership in a set of candidates.

**Inputs:** `v` and candidate values 10, 25, and 40.

**Task:** Keep entries of `v` that belong to the candidate tensor.

**Ingredients:** `torch.isin` to build the mask, then Boolean indexing.

**Output:** Assign the result to `ex054`. Produce shape `(3,)`.

**Next concept:** Masked select.


In [ ]:
# Exercise 054: assign your result to `ex054`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex054", torch.tensor([10, 25, 40], dtype=torch.int64).reshape((3,)))


### Exercise 055 — Masked select

**Purpose:** Use PyTorch's dedicated mask-selection operator.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select values divisible by 11 with `torch.masked_select`.

**Ingredients:** A same-shape Boolean mask and `torch.masked_select`.

**Output:** Assign the result to `ex055`. Produce a rank-1 tensor.

**Next concept:** Broadcasted masked select.


In [ ]:
# Exercise 055: assign your result to `ex055`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex055", torch.tensor([0, 11, 22, 33], dtype=torch.int64).reshape((4,)))


### Exercise 056 — Broadcasted masked select

**Purpose:** Use a broadcastable mask in a dedicated operator.

**Inputs:** `a` and `col_mask`; reshape the mask to `(1, 6)`.

**Task:** Use `torch.masked_select` to return all values from the selected columns across all rows.

**Ingredients:** A broadcastable Boolean mask and `torch.masked_select`; its output is always rank 1.

**Output:** Assign the result to `ex056`. Produce a rank-1 tensor with 12 entries.

**Next concept:** Conditional replacement with where.


In [ ]:
# Exercise 056: assign your result to `ex056`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex056", torch.tensor([1, 4, 5, 11, 14, 15, 21, 24, 25, 31, 34, 35], dtype=torch.int64).reshape((12,)))


### Exercise 057 — Conditional replacement with where

**Purpose:** Keep matrix shape while choosing between two values at each position.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Keep even entries unchanged and replace odd entries with `-1`, without mutating `a`.

**Ingredients:** `torch.where(condition, input, other)`.

**Output:** Assign the result to `ex057`. Produce shape `(4, 6)`.

**Next concept:** Coordinates from nonzero.


In [ ]:
# Exercise 057: assign your result to `ex057`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex057", torch.tensor([0, -1, 2, -1, 4, -1, 10, -1, 12, -1, 14, -1, 20, -1, 22, -1, 24, -1, 30, -1, 32, -1, 34, -1], dtype=torch.int64).reshape((4, 6)))


### Exercise 058 — Coordinates from nonzero

**Purpose:** Convert a Boolean condition into explicit row-column coordinates.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Find coordinates of values divisible by 11, including zero, using `torch.nonzero` with a single coordinate matrix result.

**Ingredients:** `torch.nonzero(..., as_tuple=False)`.

**Output:** Assign the result to `ex058`. Produce an integer tensor of shape `(4, 2)`.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 058: assign your result to `ex058`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex058", torch.tensor([0, 0, 1, 1, 2, 2, 3, 3], dtype=torch.int64).reshape((4, 2)))


## 6. Indexed assignment, broadcasting, views, and copies

Indexing can appear on the left side of assignment. Basic slices are views and share storage with their source; advanced-index results are copies. Every exercise in this section must begin from `a.clone()` so the supplied `a` remains unchanged.


### Exercise 059 — Assign one cell

**Purpose:** Update one coordinate on an independent working tensor.

**Inputs:** `a`; begin from a clone.

**Task:** Set row 1, column 2 to `-99` and return the modified clone.

**Ingredients:** `Tensor.clone`, two integer indices, and indexed assignment.

**Output:** Assign the result to `ex059`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Fill one row.


In [ ]:
# Exercise 059: assign your result to `ex059`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex059", torch.tensor([0, 1, 2, 3, 4, 5, 10, 11, -99, 13, 14, 15, 20, 21, 22, 23, 24, 25, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 060 — Fill one row

**Purpose:** Broadcast a scalar across an indexed row.

**Inputs:** `a`; begin from a clone.

**Task:** Set every entry of row 2 to `7`.

**Ingredients:** An integer row index and scalar assignment.

**Output:** Assign the result to `ex060`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Assign a vector into a slice.


In [ ]:
# Exercise 060: assign your result to `ex060`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex060", torch.tensor([0, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15, 7, 7, 7, 7, 7, 7, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 061 — Assign a vector into a slice

**Purpose:** Match a source vector to a one-dimensional destination slice.

**Inputs:** `a`; begin from a clone; source values are 100, 101, and 102.

**Task:** Replace row 0, columns 1 through 3 with the three source values.

**Ingredients:** A bounded slice on the left and a length-3 tensor on the right.

**Output:** Assign the result to `ex061`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Assign one value per row.


In [ ]:
# Exercise 061: assign your result to `ex061`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex061", torch.tensor([0, 100, 101, 102, 4, 5, 10, 11, 12, 13, 14, 15, 20, 21, 22, 23, 24, 25, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 062 — Assign one value per row

**Purpose:** Align a source vector with a column destination.

**Inputs:** `a`; begin from a clone; source values are 6, 16, 26, and 36.

**Task:** Replace the final column with one supplied value per row.

**Ingredients:** A full row slice, one column index, and a length-4 source tensor.

**Output:** Assign the result to `ex062`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Fill a rectangular slice.


In [ ]:
# Exercise 062: assign your result to `ex062`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex062", torch.tensor([0, 1, 2, 3, 4, 6, 10, 11, 12, 13, 14, 16, 20, 21, 22, 23, 24, 26, 30, 31, 32, 33, 34, 36], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 063 — Fill a rectangular slice

**Purpose:** Broadcast one scalar over a two-dimensional destination.

**Inputs:** `a`; begin from a clone.

**Task:** Set rows 1 and 2, columns 2 through 4, to `-1`.

**Ingredients:** Two basic slices on the left and scalar assignment.

**Output:** Assign the result to `ex063`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Assign through a Boolean mask.


In [ ]:
# Exercise 063: assign your result to `ex063`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex063", torch.tensor([0, 1, 2, 3, 4, 5, 10, 11, -1, -1, -1, 15, 20, 21, -1, -1, -1, 25, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 064 — Assign through a Boolean mask

**Purpose:** Modify every position satisfying a condition.

**Inputs:** `a`; begin from a clone.

**Task:** Replace every even entry with `-2`.

**Ingredients:** A same-shape Boolean mask on the left side of assignment.

**Output:** Assign the result to `ex064`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Assign sequential values through a mask.


In [ ]:
# Exercise 064: assign your result to `ex064`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex064", torch.tensor([-2, 1, -2, 3, -2, 5, -2, 11, -2, 13, -2, 15, -2, 21, -2, 23, -2, 25, -2, 31, -2, 33, -2, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 065 — Assign sequential values through a mask

**Purpose:** Match a rank-1 source to the true positions of a full mask.

**Inputs:** `a`; begin from a clone; there are nine original values greater than 22.

**Task:** Replace those positions, in mask traversal order, with integers 100 through 108.

**Ingredients:** A full Boolean mask and a source tensor whose length equals the number of true positions.

**Output:** Assign the result to `ex065`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Assign paired coordinates.


In [ ]:
# Exercise 065: assign your result to `ex065`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex065", torch.tensor([0, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15, 20, 21, 22, 100, 101, 102, 103, 104, 105, 106, 107, 108], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 066 — Assign paired coordinates

**Purpose:** Write one source value at each paired row-column coordinate.

**Inputs:** `a`, `row_ids`, and `col_ids`; begin from a clone; source values are -1, -2, and -3.

**Task:** Assign corresponding source values to the three paired coordinates.

**Ingredients:** Two advanced indices on the left and a length-3 source tensor.

**Output:** Assign the result to `ex066`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Replace selected rows.


In [ ]:
# Exercise 066: assign your result to `ex066`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex066", torch.tensor([0, 1, 2, 3, 4, -2, 10, 11, 12, 13, 14, 15, 20, 21, 22, -3, 24, 25, 30, -1, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 067 — Replace selected rows

**Purpose:** Assign a rank-2 source through an advanced row index.

**Inputs:** `a`; begin from a clone; replace rows 0 and 3 with rows filled with 9 and 8 respectively.

**Task:** Write the two replacement rows in one advanced-index assignment.

**Ingredients:** A rank-1 row index and a `(2, 6)` source tensor.

**Output:** Assign the result to `ex067`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Broadcast a column source.


In [ ]:
# Exercise 067: assign your result to `ex067`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex067", torch.tensor([9, 9, 9, 9, 9, 9, 10, 11, 12, 13, 14, 15, 20, 21, 22, 23, 24, 25, 8, 8, 8, 8, 8, 8], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 068 — Broadcast a column source

**Purpose:** Use assignment broadcasting over destination columns.

**Inputs:** `a`; begin from a clone; row values are 100, 200, 300, and 400 shaped as `(4, 1)`.

**Task:** Assign that column-shaped source into the first two columns so each row value repeats twice.

**Ingredients:** A `(4, 2)` destination slice and a broadcastable `(4, 1)` source.

**Output:** Assign the result to `ex068`. Produce shape `(4, 6)` without changing `a`.

**Next concept:** Mutate through a basic-slice view.


In [ ]:
# Exercise 068: assign your result to `ex068`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex068", torch.tensor([100, 100, 2, 3, 4, 5, 200, 200, 12, 13, 14, 15, 300, 300, 22, 23, 24, 25, 400, 400, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 069 — Mutate through a basic-slice view

**Purpose:** Observe that changing a basic-slice view changes its source clone.

**Inputs:** `a`; clone it into the required output, then take every second column as a view.

**Task:** Add 1000 in-place to the view and return the source clone as `ex069`.

**Ingredients:** A basic strided slice and an in-place operation on that slice.

**Output:** Assign the result to `ex069`. Produce the source clone with columns 0, 2, and 4 increased; keep `a` unchanged.

**Next concept:** Mutate an advanced-index copy.


In [ ]:
# Exercise 069: assign your result to `ex069`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex069", torch.tensor([1000, 1, 1002, 3, 1004, 5, 1010, 11, 1012, 13, 1014, 15, 1020, 21, 1022, 23, 1024, 25, 1030, 31, 1032, 33, 1034, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 070 — Mutate an advanced-index copy

**Purpose:** Contrast advanced-index copy semantics with a basic view.

**Inputs:** `a`; clone it, select columns 0, 2, and 4 using an integer tensor, and mutate that selection.

**Task:** Add 1000 in-place to the selected tensor, then return the original clone as `ex070`.

**Ingredients:** Advanced column indexing creates a copy, so mutating the selection must not affect its source.

**Output:** Assign the result to `ex070`. Produce a tensor still equal to `a`; keep `a` unchanged.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 070: assign your result to `ex070`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex070", torch.tensor([0, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15, 20, 21, 22, 23, 24, 25, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


## 7. Dedicated PyTorch indexing operators

Bracket syntax is only part of PyTorch indexing. These exercises cover operators with explicit dimension arguments. Read each operator's shape contract before coding: `index_select` uses a rank-1 index, `gather` uses an index tensor aligned to the output shape, `masked_select` returns rank 1, and scatter-family operators write in the opposite direction.


### Exercise 071 — Select a row with torch.select

**Purpose:** Use an explicit dimension-oriented alternative to an integer index.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select row 2 with `torch.select`.

**Ingredients:** `torch.select(input, dim, index)` with dimension 0.

**Output:** Assign the result to `ex071`. Produce shape `(6,)`.

**Next concept:** Select a column with torch.select.


In [ ]:
# Exercise 071: assign your result to `ex071`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex071", torch.tensor([20, 21, 22, 23, 24, 25], dtype=torch.int64).reshape((6,)))


### Exercise 072 — Select a column with torch.select

**Purpose:** Choose along dimension 1 explicitly.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select column 4 with `torch.select`.

**Ingredients:** `torch.select` with dimension 1.

**Output:** Assign the result to `ex072`. Produce shape `(4,)`.

**Next concept:** Narrow the row axis.


In [ ]:
# Exercise 072: assign your result to `ex072`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex072", torch.tensor([4, 14, 24, 34], dtype=torch.int64).reshape((4,)))


### Exercise 073 — Narrow the row axis

**Purpose:** Take a contiguous interval using start and length rather than stop.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select two rows beginning at row 1 with `torch.narrow`.

**Ingredients:** `torch.narrow(input, dim, start, length)` on dimension 0.

**Output:** Assign the result to `ex073`. Produce shape `(2, 6)`.

**Next concept:** Narrow the column axis.


In [ ]:
# Exercise 073: assign your result to `ex073`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex073", torch.tensor([10, 11, 12, 13, 14, 15, 20, 21, 22, 23, 24, 25], dtype=torch.int64).reshape((2, 6)))


### Exercise 074 — Narrow the column axis

**Purpose:** Apply start-and-length selection to dimension 1.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select three columns beginning at column 2 with `torch.narrow`.

**Ingredients:** `torch.narrow` on dimension 1.

**Output:** Assign the result to `ex074`. Produce shape `(4, 3)`.

**Next concept:** Index-select rows.


In [ ]:
# Exercise 074: assign your result to `ex074`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex074", torch.tensor([2, 3, 4, 12, 13, 14, 22, 23, 24, 32, 33, 34], dtype=torch.int64).reshape((4, 3)))


### Exercise 075 — Index-select rows

**Purpose:** Select and repeat entries along one named dimension.

**Inputs:** `a`; desired row order is 3, 1, 1.

**Task:** Use `torch.index_select` along dimension 0.

**Ingredients:** `torch.index_select(input, dim, index)` with a rank-1 integer index.

**Output:** Assign the result to `ex075`. Produce shape `(3, 6)`.

**Next concept:** Index-select columns.


In [ ]:
# Exercise 075: assign your result to `ex075`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex075", torch.tensor([30, 31, 32, 33, 34, 35, 10, 11, 12, 13, 14, 15, 10, 11, 12, 13, 14, 15], dtype=torch.int64).reshape((3, 6)))


### Exercise 076 — Index-select columns

**Purpose:** Use the same operator on the column dimension.

**Inputs:** `a`; desired column order is 4, 0, 2.

**Task:** Use `torch.index_select` along dimension 1.

**Ingredients:** A rank-1 integer index and dimension 1.

**Output:** Assign the result to `ex076`. Produce shape `(4, 3)`.

**Next concept:** Gather one column per row.


In [ ]:
# Exercise 076: assign your result to `ex076`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex076", torch.tensor([4, 0, 2, 14, 10, 12, 24, 20, 22, 34, 30, 32], dtype=torch.int64).reshape((4, 3)))


### Exercise 077 — Gather one column per row

**Purpose:** Express paired row-wise selection with `gather`.

**Inputs:** `a` and `target_cols`, one target column per row.

**Task:** Gather along dimension 1 while keeping a singleton gathered-column dimension.

**Ingredients:** `torch.gather`; reshape or unsqueeze the index to `(4, 1)`.

**Output:** Assign the result to `ex077`. Produce shape `(4, 1)`.

**Next concept:** Gather then remove the singleton.


In [ ]:
# Exercise 077: assign your result to `ex077`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex077", torch.tensor([2, 10, 25, 31], dtype=torch.int64).reshape((4, 1)))


### Exercise 078 — Gather then remove the singleton

**Purpose:** Return the familiar one-value-per-row vector after `gather`.

**Inputs:** `a` and `target_cols`.

**Task:** Gather one column per row, then remove only the size-1 gathered dimension.

**Ingredients:** `torch.gather` followed by a dimension-specific squeeze.

**Output:** Assign the result to `ex078`. Produce shape `(4,)`.

**Next concept:** Gather two columns per row.


In [ ]:
# Exercise 078: assign your result to `ex078`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex078", torch.tensor([2, 10, 25, 31], dtype=torch.int64).reshape((4,)))


### Exercise 079 — Gather two columns per row

**Purpose:** Use a different pair of column indices for every row.

**Inputs:** `a` and `gather_cols`, shape `(4, 2)`.

**Task:** Gather from dimension 1 using the complete index matrix.

**Ingredients:** `torch.gather(input, dim=1, index=...)`.

**Output:** Assign the result to `ex079`. Produce shape `(4, 2)`.

**Next concept:** Gather features from a rank-3 tensor.


In [ ]:
# Exercise 079: assign your result to `ex079`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex079", torch.tensor([0, 5, 11, 14, 22, 23, 35, 30], dtype=torch.int64).reshape((4, 2)))


### Exercise 080 — Gather features from a rank-3 tensor

**Purpose:** Generalize `gather` to higher rank.

**Inputs:** `cube`, shape `(2, 3, 4)`, and `cube_feature_idx`, shape `(2, 3, 2)`.

**Task:** For every batch-channel position, gather the two requested feature entries along dimension 2.

**Ingredients:** `torch.gather` with input and index aligned on non-gather dimensions.

**Output:** Assign the result to `ex080`. Produce shape `(2, 3, 2)`.

**Next concept:** Take along a dimension.


In [ ]:
# Exercise 080: assign your result to `ex080`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex080", torch.tensor([0, 3, 5, 6, 10, 8, 15, 13, 16, 18, 21, 23], dtype=torch.int64).reshape((2, 3, 2)))


### Exercise 081 — Take along a dimension

**Purpose:** Reorder each row with its own permutation.

**Inputs:** `scores`, shape `(4, 5)`, and `score_order`, the descending column order for each row.

**Task:** Use `torch.take_along_dim` to reorder each score row from largest to smallest.

**Ingredients:** `torch.take_along_dim(input, indices, dim=1)`.

**Output:** Assign the result to `ex081`. Produce shape `(4, 5)`.

**Next concept:** Take from flattened storage order.


In [ ]:
# Exercise 081: assign your result to `ex081`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex081", torch.tensor([2.200000047683716, 1.2999999523162842, 0.20000000298023224, 0.0, -1.0, 2.0, 1.100000023841858, 1.0, 0.5, -0.30000001192092896, 3.0, 0.10000000149011612, 0.0, -0.10000000149011612, -1.0, 2.4000000953674316, 1.399999976158142, 0.699999988079071, 0.30000001192092896, -2.0], dtype=torch.float32).reshape((4, 5)))


### Exercise 082 — Take from flattened storage order

**Purpose:** Select by flat logical positions regardless of rank.

**Inputs:** `a` and `flat_take = [0, 7, 14, 23]`.

**Task:** Use `torch.take` to select those flattened positions.

**Ingredients:** `torch.take(input, index)`.

**Output:** Assign the result to `ex082`. Produce shape `(4,)`.

**Next concept:** Masked-select with broadcasting.


In [ ]:
# Exercise 082: assign your result to `ex082`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex082", torch.tensor([0, 11, 22, 35], dtype=torch.int64).reshape((4,)))


### Exercise 083 — Masked-select with broadcasting

**Purpose:** Reinforce the dedicated mask operator's broadcasting and output-rank rules.

**Inputs:** `a` and `col_mask`.

**Task:** Reshape `col_mask` to broadcast across rows, then call `torch.masked_select`.

**Ingredients:** `None` for a singleton mask axis and `torch.masked_select`.

**Output:** Assign the result to `ex083`. Produce a rank-1 tensor with 12 entries.

**Next concept:** Accumulating index_put.


In [ ]:
# Exercise 083: assign your result to `ex083`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex083", torch.tensor([1, 4, 5, 11, 14, 15, 21, 24, 25, 31, 34, 35], dtype=torch.int64).reshape((12,)))


### Exercise 084 — Accumulating index_put

**Purpose:** Handle repeated write indices with defined summation behavior.

**Inputs:** A new length-5 zero integer tensor; indices 1, 1, and 3; values 2, 5, and 7.

**Task:** Use in-place `index_put_` with `accumulate=True` so repeated index 1 receives both contributions.

**Ingredients:** A one-element tuple of index tensors, a values tensor, and the accumulate flag.

**Output:** Assign the result to `ex084`. Produce shape `(5,)`.

**Next concept:** Scatter one value per row.


In [ ]:
# Exercise 084: assign your result to `ex084`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex084", torch.tensor([0, 7, 0, 7, 0], dtype=torch.int64).reshape((5,)))


### Exercise 085 — Scatter one value per row

**Purpose:** Reverse the direction of gather by writing sources into indexed destinations.

**Inputs:** A `(4, 6)` integer zero tensor, `target_cols`, and source values 9, 8, 7, and 6.

**Task:** Place one source value in each row at that row's target column.

**Ingredients:** `Tensor.scatter` along dimension 1; index and source should both have shape `(4, 1)`.

**Output:** Assign the result to `ex085`. Produce shape `(4, 6)`.

**Next concept:** Scatter-add repeated destinations.


In [ ]:
# Exercise 085: assign your result to `ex085`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex085", torch.tensor([0, 0, 9, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 6, 0, 0, 0, 0], dtype=torch.int64).reshape((4, 6)))


### Exercise 086 — Scatter-add repeated destinations

**Purpose:** Accumulate multiple sources that map to the same destination.

**Inputs:** A `(2, 5)` integer zero tensor; index rows `[0,1,1,3]` and `[2,2,4,2]`; source rows `[1,2,3,4]` and `[5,6,7,8]`.

**Task:** Use `scatter_add` along dimension 1.

**Ingredients:** Index and source tensors of identical shape plus a destination dimension.

**Output:** Assign the result to `ex086`. Produce shape `(2, 5)` with repeated destinations summed.

**Next concept:** Index-fill selected rows.


In [ ]:
# Exercise 086: assign your result to `ex086`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex086", torch.tensor([1, 5, 0, 4, 0, 0, 0, 19, 0, 7], dtype=torch.int64).reshape((2, 5)))


### Exercise 087 — Index-fill selected rows

**Purpose:** Fill complete entries along a named dimension.

**Inputs:** `a`; selected row indices are 1 and 3; fill value is `-5`.

**Task:** Return a modified copy made with `index_fill`, leaving `a` untouched.

**Ingredients:** `Tensor.index_fill(dim, index, value)` on dimension 0.

**Output:** Assign the result to `ex087`. Produce shape `(4, 6)`.

**Next concept:** Index-copy source rows.


In [ ]:
# Exercise 087: assign your result to `ex087`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex087", torch.tensor([0, 1, 2, 3, 4, 5, -5, -5, -5, -5, -5, -5, 20, 21, 22, 23, 24, 25, -5, -5, -5, -5, -5, -5], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)


### Exercise 088 — Index-copy source rows

**Purpose:** Copy whole source slices into chosen destination positions.

**Inputs:** A zero tensor shaped like `a`; destination rows 0 and 2; source rows are all 7s and all 9s.

**Task:** Use `index_copy` along dimension 0.

**Ingredients:** Destination tensor, dimension, rank-1 destination index, and `(2, 6)` source.

**Output:** Assign the result to `ex088`. Produce shape `(4, 6)`.

**Next concept:** Index-add grouped rows.


In [ ]:
# Exercise 088: assign your result to `ex088`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex088", torch.tensor([7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0], dtype=torch.int64).reshape((4, 6)))


### Exercise 089 — Index-add grouped rows

**Purpose:** Aggregate source rows into destination groups.

**Inputs:** A `(3, 2)` integer zero tensor; group indices `[0,1,0,2]`; source rows `[[1,2],[3,4],[5,6],[7,8]]`.

**Task:** Use `index_add` along dimension 0 so both group-0 rows are summed.

**Ingredients:** Destination tensor, dimension, rank-1 group index, and aligned source rows.

**Output:** Assign the result to `ex089`. Produce shape `(3, 2)`.

**Next concept:** Diagonal operator.


In [ ]:
# Exercise 089: assign your result to `ex089`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex089", torch.tensor([6, 8, 3, 4, 7, 8], dtype=torch.int64).reshape((3, 2)))


### Exercise 090 — Diagonal operator

**Purpose:** Use a dedicated view-producing diagonal selector.

**Inputs:** The first four columns of `a`, forming a `(4, 4)` matrix.

**Task:** Extract its main diagonal with `torch.diagonal`.

**Ingredients:** A basic slice followed by `torch.diagonal` with the two matrix dimensions.

**Output:** Assign the result to `ex090`. Produce shape `(4,)`.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 090: assign your result to `ex090`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex090", torch.tensor([0, 11, 22, 33], dtype=torch.int64).reshape((4,)))


## 8. Machine-learning indexing patterns

Here `scores` contains four training examples and five class candidates per example. `class_targets[i]` is the expected class for example `i`. After `log_softmax`, only the indexed probability at that expected class contributes directly to that example's negative log-likelihood.

For one example, the loss is

$$
\ell_i = -\log p_{i,y_i}
$$

Here `i` identifies a training example, `y_i` is its expected target class, and `p_{i,y_i}` is the model probability assigned to that target among all class candidates.

For all `N` examples, the dataset-average loss is

$$
L = \frac{1}{N}\sum_{i=1}^{N} \ell_i
$$

Here `N` is the number of scored examples and `L` is the average, not the total. Exercises 091–098 score the full supplied four-example toy dataset. For token data, each batch-position pair is one training example and the final axis contains five vocabulary candidates; Exercise 099 scores all six supplied positions.


### Exercise 091 — Target score per example

**Purpose:** Apply paired indexing to a classification batch.

**Inputs:** `scores`, shape `(4, 5)`, and `class_targets`, shape `(4,)`.

**Task:** Select the raw score at each example's expected target class using a row range and the target tensor.

**Ingredients:** `range(n)` on the example axis and `class_targets` on the candidate axis.

**Output:** Assign the result to `ex091`. Produce shape `(4,)`, one selected target score per training example.

**Next concept:** Target log-probability per example.


In [ ]:
# Exercise 091: assign your result to `ex091`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex091", torch.tensor([2.200000047683716, 1.100000023841858, 3.0, 1.399999976158142], dtype=torch.float32).reshape((4,)))


### Exercise 092 — Target log-probability per example

**Purpose:** Select the probability that contributes directly to each example's loss.

**Inputs:** `scores` and `class_targets`.

**Task:** Compute log-probabilities across the five class candidates, then select each example's expected target log-probability with paired indexing.

**Ingredients:** `Tensor.log_softmax(dim=1)` and one-example-per-row advanced indexing.

**Output:** Assign the result to `ex092`. Produce shape `(4,)`.

**Next concept:** Per-example negative log-likelihood.


In [ ]:
# Exercise 092: assign your result to `ex092`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex092", torch.tensor([-0.52677983045578, -1.6409072875976562, -0.15544278919696808, -1.5219416618347168], dtype=torch.float32).reshape((4,)))


### Exercise 093 — Per-example negative log-likelihood

**Purpose:** Turn selected target log-probabilities into individual losses.

**Inputs:** `scores` and `class_targets` for all four supplied examples.

**Task:** Compute one negative log-likelihood per example; do not reduce across examples.

**Ingredients:** Log-softmax, paired target indexing, and unary negation.

**Output:** Assign the result to `ex093`. Produce shape `(4,)`; each entry is a loss for one training example.

**Next concept:** Average negative log-likelihood.


In [ ]:
# Exercise 093: assign your result to `ex093`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex093", torch.tensor([0.52677983045578, 1.6409072875976562, 0.15544278919696808, 1.5219416618347168], dtype=torch.float32).reshape((4,)))


### Exercise 094 — Average negative log-likelihood

**Purpose:** Reduce the full supplied dataset to its average loss.

**Inputs:** All four rows of `scores` and all four `class_targets`.

**Task:** Compute the mean of the four per-example negative log-likelihoods.

**Ingredients:** Log-softmax, paired target indexing, negation, and `Tensor.mean()`.

**Output:** Assign the result to `ex094`. Produce one rank-0 floating-point tensor representing an average, not a total.

**Next concept:** Built-in cross-entropy check.


In [ ]:
# Exercise 094: assign your result to `ex094`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex094", torch.tensor([0.96126788854599], dtype=torch.float32).reshape(()))


### Exercise 095 — Built-in cross-entropy check

**Purpose:** Connect manual indexing to PyTorch's combined loss operator.

**Inputs:** All four rows of `scores` and `class_targets`.

**Task:** Compute the default mean cross-entropy with `torch.nn.functional.cross_entropy`.

**Ingredients:** `F.cross_entropy(logits, targets)`; pass raw scores, not log-probabilities.

**Output:** Assign the result to `ex095`. Produce one rank-0 floating-point tensor equal to Exercise 094.

**Next concept:** Predicted class indices.


In [ ]:
# Exercise 095: assign your result to `ex095`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex095", torch.tensor([0.96126788854599], dtype=torch.float32).reshape(()))


### Exercise 096 — Predicted class indices

**Purpose:** Use reduction indices as model predictions.

**Inputs:** `scores`, shape `(4, 5)`.

**Task:** Find the highest-scoring class index for every example.

**Ingredients:** `Tensor.argmax` along the class-candidate dimension.

**Output:** Assign the result to `ex096`. Produce a length-4 integer tensor.

**Next concept:** Correctness mask.


In [ ]:
# Exercise 096: assign your result to `ex096`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex096", torch.tensor([3, 0, 4, 1], dtype=torch.int64).reshape((4,)))


### Exercise 097 — Correctness mask

**Purpose:** Compare one prediction with one expected target per example.

**Inputs:** `scores` and `class_targets`.

**Task:** Return a Boolean tensor saying whether each row's highest-scoring class equals its target.

**Ingredients:** Argmax along candidates followed by elementwise equality.

**Output:** Assign the result to `ex097`. Produce a length-4 Boolean tensor.

**Next concept:** Target class weight per example.


In [ ]:
# Exercise 097: assign your result to `ex097`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex097", torch.tensor([True, False, True, False], dtype=torch.bool).reshape((4,)))


### Exercise 098 — Target class weight per example

**Purpose:** Look up metadata using target class indices.

**Inputs:** `class_weights`, length 5, and `class_targets`, length 4.

**Task:** Select the weight belonging to each example's expected target class.

**Ingredients:** Use the target tensor as an advanced index into the class-weight vector.

**Output:** Assign the result to `ex098`. Produce a length-4 floating-point tensor.

**Next concept:** Target token log-probabilities.


In [ ]:
# Exercise 098: assign your result to `ex098`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex098", torch.tensor([2.0, 3.0, 3.0, 0.5], dtype=torch.float32).reshape((4,)))


### Exercise 099 — Target token log-probabilities

**Purpose:** Extend paired indexing to batch, sequence-position, and vocabulary axes.

**Inputs:** `token_logits`, shape `(2, 3, 5)`, and `token_targets`, shape `(2, 3)`.

**Task:** Compute log-probabilities over the five vocabulary candidates, then select the expected target at every one of the six batch-position training examples without loops.

**Ingredients:** Broadcastable batch and position index grids plus `token_targets` on the vocabulary axis.

**Output:** Assign the result to `ex099`. Produce shape `(2, 3)`; each selected target probability contributes directly to its position's loss.

**Next concept:** Embedding lookup.


In [ ]:
# Exercise 099: assign your result to `ex099`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex099", torch.tensor([-1.9042471647262573, -1.628385066986084, -1.3525229692459106, -1.7663161754608154, -1.4904539585113525, -1.9042471647262573], dtype=torch.float32).reshape((2, 3)))


### Exercise 100 — Embedding lookup

**Purpose:** Recognize embedding lookup as advanced indexing into a table.

**Inputs:** `embedding_table`, shape `(8, 3)`, and `token_ids`, shape `(2, 4)`.

**Task:** Retrieve the three-dimensional embedding vector for every token ID with direct tensor indexing.

**Ingredients:** Use the rank-2 integer ID tensor as an advanced index into axis 0 of the table.

**Output:** Assign the result to `ex100`. Produce shape `(2, 4, 3)`.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 100: assign your result to `ex100`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex100", torch.tensor([0.6000000238418579, 0.699999988079071, 0.800000011920929, 1.5, 1.600000023841858, 1.7000000476837158, 0.30000001192092896, 0.4000000059604645, 0.5, 0.0, 0.10000000149011612, 0.20000000298023224, 2.0999999046325684, 2.200000047683716, 2.299999952316284, 0.8999999761581421, 1.0, 1.100000023841858, 0.8999999761581421, 1.0, 1.100000023841858, 1.2000000476837158, 1.2999999523162842, 1.399999976158142], dtype=torch.float32).reshape((2, 4, 3)))


## 9. Storage and autograd semantics

Basic indexing returns a view that shares storage; advanced indexing returns a copy. During backpropagation, gradients flow only to selected source positions. If an index is repeated, contributions to that source position accumulate.


### Exercise 101 — Basic slice shares storage

**Purpose:** Verify view semantics rather than only matching values.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select columns 1 through 3 with a basic slice and return that selection directly.

**Ingredients:** A basic column slice; do not clone the result.

**Output:** Assign the result to `ex101`. Produce shape `(4, 3)` and share storage with `a`.

**Next concept:** Advanced selection owns separate storage.


In [ ]:
# Exercise 101: assign your result to `ex101`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex101", torch.tensor([1, 2, 3, 11, 12, 13, 21, 22, 23, 31, 32, 33], dtype=torch.int64).reshape((4, 3)))
_check_storage("ex101", a, True)


### Exercise 102 — Advanced selection owns separate storage

**Purpose:** Verify copy semantics for integer-tensor indexing.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Select columns 1, 2, and 3 with an integer tensor and return the result directly.

**Ingredients:** Advanced column indexing; do not clone because the indexing operation itself must create the copy.

**Output:** Assign the result to `ex102`. Produce shape `(4, 3)` and not share storage with `a`.

**Next concept:** Clone owns separate storage.


In [ ]:
# Exercise 102: assign your result to `ex102`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex102", torch.tensor([1, 2, 3, 11, 12, 13, 21, 22, 23, 31, 32, 33], dtype=torch.int64).reshape((4, 3)))
_check_storage("ex102", a, False)


### Exercise 103 — Clone owns separate storage

**Purpose:** Make an explicit independent copy before mutation.

**Inputs:** `a`, shape `(4, 6)`.

**Task:** Clone `a`, set the clone's top-left entry to `-1`, and return the clone.

**Ingredients:** `Tensor.clone` followed by indexed assignment.

**Output:** Assign the result to `ex103`. Produce shape `(4, 6)`, not share storage with `a`, and leave `a` unchanged.

**Next concept:** Gradient through unique selected positions.


In [ ]:
# Exercise 103: assign your result to `ex103`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex103", torch.tensor([-1, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15, 20, 21, 22, 23, 24, 25, 30, 31, 32, 33, 34, 35], dtype=torch.int64).reshape((4, 6)))
_check_tensor("a", _A_CANONICAL)
_check_storage("ex103", a, False)


### Exercise 104 — Gradient through unique selected positions

**Purpose:** See that unselected source entries receive zero gradient.

**Inputs:** Create a floating vector containing 0 through 4 with `requires_grad=True`.

**Task:** Select positions 0, 2, and 4, sum them, backpropagate, and assign the source gradient to `ex104`.

**Ingredients:** Advanced integer indexing, `sum`, and `backward`.

**Output:** Assign the result to `ex104`. Produce a length-5 floating gradient tensor.

**Next concept:** Gradient through a repeated index.


In [ ]:
# Exercise 104: assign your result to `ex104`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex104", torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0], dtype=torch.float32).reshape((5,)))


### Exercise 105 — Gradient through a repeated index

**Purpose:** Observe gradient accumulation at a repeated source position.

**Inputs:** Create a floating vector containing 0 through 4 with `requires_grad=True`.

**Task:** Select positions 1, 1, and 3, sum them, backpropagate, and return the source gradient.

**Ingredients:** An advanced index with a duplicate, `sum`, and `backward`.

**Output:** Assign the result to `ex105`. Produce a length-5 gradient tensor; the repeated source receives two contributions.

**Next concept:** Gradient through gather with repeats.


In [ ]:
# Exercise 105: assign your result to `ex105`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex105", torch.tensor([0.0, 2.0, 0.0, 1.0, 0.0], dtype=torch.float32).reshape((5,)))


### Exercise 106 — Gradient through gather with repeats

**Purpose:** Generalize repeated-index gradient accumulation to a rank-2 gather.

**Inputs:** Create a `(2, 3)` floating tensor containing 0 through 5 as a leaf requiring gradients; gather indices are `[0,0]` for row 0 and `[1,2]` for row 1.

**Task:** Gather along columns, sum all gathered values, backpropagate, and return the source gradient.

**Ingredients:** A leaf tensor, `torch.gather`, `sum`, and `backward`.

**Output:** Assign the result to `ex106`. Produce shape `(2, 3)`; repeated destinations in the backward pass accumulate.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 106: assign your result to `ex106`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex106", torch.tensor([2.0, 0.0, 0.0, 0.0, 1.0, 1.0], dtype=torch.float32).reshape((2, 3)))


## 10. Vectorized mastery capstones

These final tasks combine several indexing ideas. Solve each without Python loops. The last loss evaluates the complete supplied token batch: two sequences times three positions, for six training examples total.


### Exercise 107 — Batched diagonals without a loop

**Purpose:** Extract one diagonal from every matrix in a batch with advanced indexing.

**Inputs:** `batch_mats`, shape `(2, 4, 4)`.

**Task:** Use one shared position index to extract both 4-by-4 main diagonals at once. Do not use `torch.diagonal` or a Python loop.

**Ingredients:** A full batch slice plus paired row and column index tensors.

**Output:** Assign the result to `ex107`. Produce shape `(2, 4)`.

**Next concept:** Variable-length validity mask.


In [ ]:
# Exercise 107: assign your result to `ex107`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex107", torch.tensor([0, 5, 10, 15, 16, 21, 26, 31], dtype=torch.int64).reshape((2, 4)))


### Exercise 108 — Variable-length validity mask

**Purpose:** Build a broadcast mask from sequence lengths and use it to remove padding.

**Inputs:** `padded`, shape `(3, 5)`, and `lengths = [3, 4, 2]`.

**Task:** Compare broadcast position indices with each row's length, then return every valid non-padding value in row-major order. Do not loop.

**Ingredients:** `torch.arange`, singleton-axis insertion, broadcasting, comparison, and Boolean indexing.

**Output:** Assign the result to `ex108`. Produce a rank-1 integer tensor containing all nine valid positions.

**Next concept:** Confusion matrix by accumulated indexing.


In [ ]:
# Exercise 108: assign your result to `ex108`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex108", torch.tensor([4, 5, 6, 7, 8, 9, 10, 11, 12], dtype=torch.int64).reshape((9,)))


### Exercise 109 — Confusion matrix by accumulated indexing

**Purpose:** Count paired true-predicted class coordinates.

**Inputs:** `true_classes`, `pred_classes`, and three possible classes.

**Task:** Create a 3-by-3 integer zero matrix and add one at row `true_classes[i]`, column `pred_classes[i]` for all seven examples. Do not loop.

**Ingredients:** Paired coordinate indices and `index_put_` with `accumulate=True`, or an equivalent scatter-add construction.

**Output:** Assign the result to `ex109`. Produce shape `(3, 3)` where rows are expected classes and columns are predicted classes.

**Next concept:** Mean token negative log-likelihood.


In [ ]:
# Exercise 109: assign your result to `ex109`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex109", torch.tensor([1, 1, 0, 0, 1, 1, 1, 0, 2], dtype=torch.int64).reshape((3, 3)))


### Exercise 110 — Mean token negative log-likelihood

**Purpose:** Combine vectorized target indexing and reduction over every supplied token position.

**Inputs:** `token_logits`, shape `(2, 3, 5)`, and `token_targets`, shape `(2, 3)`.

**Task:** Compute the mean negative log-likelihood across all six batch-position training examples without loops and without calling cross-entropy.

**Ingredients:** Log-softmax over vocabulary candidates, broadcast batch-position indices, direct target selection, negation, and mean reduction.

**Output:** Assign the result to `ex110`. Produce one rank-0 floating tensor representing the full supplied batch average, not a total and not a single test sequence.

**Next concept:** the next section or the final mastery check.


In [ ]:
# Exercise 110: assign your result to `ex110`.
# Write your solution below this comment, then run the supplied test cell.


In [ ]:
# Supplied test: run this cell after writing the exercise answer.
_check_tensor("ex110", torch.tensor([1.6743621826171875], dtype=torch.float32).reshape(()))


## Official PyTorch references

Use these after making an honest attempt; they document the contracts exercised above without providing this notebook's answers.

- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)
- [`torch.select`](https://docs.pytorch.org/docs/stable/generated/torch.select.html)
- [`torch.narrow`](https://docs.pytorch.org/docs/stable/generated/torch.narrow.html)
- [`torch.index_select`](https://docs.pytorch.org/docs/stable/generated/torch.index_select.html)
- [`torch.gather`](https://docs.pytorch.org/docs/stable/generated/torch.gather.html)
- [`torch.take_along_dim`](https://docs.pytorch.org/docs/stable/generated/torch.take_along_dim.html)
- [`torch.masked_select`](https://docs.pytorch.org/docs/stable/generated/torch.masked_select.html)
- [`Tensor.index_put_`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.index_put_.html)
- [`Tensor.scatter_`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.scatter_.html)
- [`Tensor.scatter_add_`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.scatter_add_.html)

## Completion standard

You have completed the workbook when every supplied test passes in a fresh kernel after **Restart Kernel and Run All**, with your solution code confined to student-answer cells. At that point you have practiced scalar, slice, advanced, Boolean, assignment, operator-based, batched machine-learning, storage, and gradient indexing semantics.
